In [ ]:
import os
import sys
import numpy as np
from sklearn.metrics import adjusted_mutual_info_score as ami
from sklearn.metrics import adjusted_rand_score as ari
from sklearn.metrics import normalized_mutual_info_score as nmi

from tqdm import tqdm
import glob
from mpire.pool import WorkerPool
from sklearn.metrics import pairwise_distances
import pandas as pd


HCF_ROOT_FOLDER = "/export/share/peters57dm/Verbund/deepsync/experiments/"
os.chdir(HCF_ROOT_FOLDER)
sys.path.append(HCF_ROOT_FOLDER)

from helper.datasets import (
    load_pendigits,
    load_optdigits,
    load_letterrecognition,
    load_gaussian_blobs,
    load_example,
    load_usps,
    load_htru,
    load_har,
    load_mice,
    load_synth_high,
    load_synth_low,
    load_mnist,
    load_fmnist,
    load_cifar10,
    load_coil20,
    load_coil100,
    load_cifar100,
    load_weizmann,
)
from helper.deep import (
    Autoencoder,
    detect_device,
    get_train_and_testloader,
    load_pretrained_model,
    encode_batchwise,
)

datasets_loading_methods = [
    (load_pendigits, "pendigits"),
    (load_optdigits, "optdigits"),
    (load_letterrecognition, "letterrecognition"),
    (load_gaussian_blobs, "easy_blobs"),
    (load_example, "example"),
    (load_usps, "USPS"),
    (load_htru, "htru"),
    (load_har, "HAR"),
    (load_mice, "mice"),
    (load_synth_high, "synth_high"),
    (load_synth_low, "synth_low"),
    (load_mnist, "MNIST"),
    (load_fmnist, "FMNIST"),
    (load_cifar10, "cifar10"),
    (load_coil20, "coil20"),
    (load_coil100, "coil100"),
    (load_cifar100, "cifar100"),
    (load_weizmann, "weizmann"),
]

In [2]:
# l = glob.glob("./core_pts/*")
# set([i.split("##")[1] for i in l])

In [11]:
from SHiP import SHiP
from SHiP.ultrametric_tree import UltrametricTreeType as UTreeType, AVAILABLE_ULTRAMETRIC_TREE_TYPES, ultrametricTreeTypeToString
from SHiP.partitioning import PartitioningMethod as PMethod, AVAILABLE_PARTITIONING_METHODS, partitioningMethodToString

excludeTreeTypes = [
    UTreeType.LoadTree,
]
TREE_TYPES = [treeType for treeType in AVAILABLE_ULTRAMETRIC_TREE_TYPES if treeType not in excludeTreeTypes]
TREE_TYPES = [UTreeType.DCTree]
HIERACHIES = range(0, 5)
PARTITIONING_METHODS = AVAILABLE_PARTITIONING_METHODS

In [4]:
def load_data_and_embedding(load_fn, data_name):
    savestring_data = f"./.cache/{data_name}##data.npy"
    savestring_embedding = f"./.cache/{data_name}##embedding.npy"
    savestring_gt_labels = f"./.cache/{data_name}##gt_labels.npy"

    if os.path.exists(savestring_data) and os.path.exists(savestring_embedding) and os.path.exists(savestring_gt_labels):
        return np.load(savestring_data), np.load(savestring_embedding), np.load(savestring_gt_labels)

    data, gt_labels, _, _ = load_fn()

    PRETRAINED_MODELS_ROOT_PATH = (
        "/export/share/peters57dm/Verbund/deepsync/experiments/comparison102/ae_sync_loss/knn_label_assignment"
    )
    EXP_NO_NAME = "exp_00"
    BATCH_SIZE = 256
    MAX_EMBED_SIZE = 10
    embedded_space_dim = min(data.shape[1], MAX_EMBED_SIZE)

    model = Autoencoder(input_dim=data.shape[1], embedding_size=embedded_space_dim)
    trainloader, testloader = get_train_and_testloader(data, gt_labels, BATCH_SIZE)

    pretrained_model_path = os.path.join(PRETRAINED_MODELS_ROOT_PATH, data_name, EXP_NO_NAME, 'pretrained_autoencoder.pth')
    model = load_pretrained_model(model, pretrained_model_path, device="cpu")
    embedded, gt_labels = encode_batchwise(testloader, model, device="cpu")

    os.makedirs(os.path.dirname(savestring_data), exist_ok=True)
    np.save(savestring_data, data)
    np.save(savestring_embedding, embedded)
    np.save(savestring_gt_labels, gt_labels)

    return data, embedded, gt_labels

In [5]:
def find_local_core_points_v2(data, k, percent):
    n = data.shape[0]
    subset = int(np.floor(n * percent))

    p_dist = pairwise_distances(data, metric="euclidean")
    core_dists = np.partition(p_dist, k - 1, axis=0)[k - 1]

    nn = np.argpartition(p_dist, subset, axis=1)[:, :subset]
    # nn_mask = np.tile(np.arange(n).reshape(-1, 1), (1, subset))
    # nn = nn[nn != nn_mask].reshape(n, subset - 1)
    refined_medians = np.median(core_dists[nn], axis=1)

    vector_mask = core_dists < refined_medians
    return vector_mask, refined_medians


def find_local_core_points_v3(data, k, percent):
    n = data.shape[0]
    subset = int(np.floor(n * percent))

    p_dist = pairwise_distances(data, metric="euclidean")
    core_dists = np.partition(p_dist, k - 1, axis=0)[k - 1]

    nn = np.argpartition(p_dist, subset, axis=1)[:, :subset]
    # nn_mask = np.tile(np.arange(n).reshape(-1, 1), (1, subset))
    # nn = nn[nn != nn_mask].reshape(n, subset - 1)

    refined_medians = np.empty((n))
    for i in range(n):
        p_dist_neighbors = p_dist[np.ix_(nn[i], nn[i])]
        core_dists_neighbors = np.partition(p_dist_neighbors, k - 1, axis=0)[k - 1]
        refined_medians[i] = np.median(core_dists_neighbors)

    vector_mask = core_dists < refined_medians
    return vector_mask, refined_medians

In [6]:
def get_core_pts(core_pts_name, data_name, space_name, find_core_pts_fn, data):
    savestring = f"./core_pts/{core_pts_name}##{data_name}##{space_name}.npy"
    if os.path.exists(savestring):
        return np.load(savestring)

    core_points_mask, _ = find_core_pts_fn(data, k=50, percent=0.1)

    os.makedirs(os.path.dirname(savestring), exist_ok=True)
    np.save(savestring, core_points_mask)
    return core_points_mask

In [7]:
def run_ship(core_pts_name, data_name, space_name, find_core_pts_fn, data, gt_labels, treeType):
    core_points_mask = get_core_pts(core_pts_name, data_name, space_name, find_core_pts_fn, data)
    # core_points = data[core_points_mask]
    gt_labels = gt_labels[core_points_mask]

    data_frames = []

    for power in HIERACHIES:
        for partitioningMethod in PARTITIONING_METHODS:
            savestring = f"./labels/{core_pts_name}##{data_name}##{space_name}##{treeType}##{power}##{partitioningMethod}.npy"
            if os.path.exists(savestring):
                labels = np.load(savestring)
                df_run = pd.DataFrame(
                    [
                        {
                            "core_pts_name": core_pts_name,
                            "data_name": data_name,
                            "space_name": space_name,
                            "treeType": ultrametricTreeTypeToString(treeType),
                            "power": power,
                            "partitioningMethod": partitioningMethodToString(partitioningMethod),
                            "ari": ari(gt_labels, labels),
                            "ami": ami(gt_labels, labels),
                            "nmi": nmi(gt_labels, labels),
                        }
                    ]
                )
                data_frames.append(df_run)
    return data_frames

In [13]:
pool = WorkerPool(n_jobs=20, use_dill=True)

async_results = {}

for load_fn, data_name in tqdm(datasets_loading_methods, desc="Datasets"):
    original_data, embedded_data, gt_labels = load_data_and_embedding(load_fn, data_name)
    for treeType in TREE_TYPES:
        for find_core_pts_fn, core_pts_name in [
            (find_local_core_points_v2, "same_core_pts"),
            (find_local_core_points_v3, "adaptive_core_pts"),
        ]:
            for data, space_name in [
                (original_data, "original_space"),
                (embedded_data, "embedding_space"),
            ]:
                savestring = f"./core_pts/{core_pts_name}##{data_name}##{space_name}.npy"
                if not os.path.exists(savestring):
                    continue
                async_idx = (core_pts_name, data_name, space_name, treeType)

                # async_results[async_idx] = pool.apply_async(
                #     run_ship, args=(core_pts_name, data_name, space_name, find_core_pts_fn, data, gt_labels, treeType)
                # )

                # print(data_name, core_pts_name, space_name, treeType)
                async_results[async_idx] = run_ship(core_pts_name, data_name, space_name, find_core_pts_fn, data, gt_labels, treeType)

data_frames = []
for async_idx, async_result in async_results.items():
    (core_pts_name, data_name, space_name, treeType) = async_idx
    # print(core_pts_name, data_name, space_name, treeType)

    # df_runs = async_result.get()
    df_runs = async_result
    data_frames.extend(df_runs)

df = pd.concat(data_frames, ignore_index=True)

pool.stop_and_join()
pool.terminate()

Datasets:   0%|                                                                                  | 0/18 [00:00<?, ?it/s]

Datasets: 100%|█████████████████████████████████████████████████████████████████████████| 18/18 [00:30<00:00,  1.72s/it]


In [14]:
df_backup = df
df

,core_pts_name,data_name,space_name,treeType,power,partitioningMethod,ari,ami,nmi
0,same_core_pts,pendigits,original_space,DCTree,0,K,0.783925,0.886050,0.886557
1,same_core_pts,pendigits,original_space,DCTree,0,Threshhold,0.783925,0.886050,0.886557
2,same_core_pts,pendigits,original_space,DCTree,0,ThreshholdElbow,0.941955,0.945112,0.945821
3,same_core_pts,pendigits,original_space,DCTree,0,QCoverage,0.946653,0.952101,0.952331
4,same_core_pts,pendigits,original_space,DCTree,0,QCoverageElbow,0.000601,0.121596,0.138495
...,...,...,...,...,...,...,...,...,...
3405,adaptive_core_pts,weizmann,embedding_space,DCTree,4,QStemElbow,0.290703,0.663641,0.688566
3406,adaptive_core_pts,weizmann,embedding_space,DCTree,4,Stability,0.525548,0.753195,0.800810
3407,adaptive_core_pts,weizmann,embedding_space,DCTree,4,ElbowOld,0.067375,0.459548,0.478506
3408,adaptive_core_pts,weizmann,embedding_space,DCTree,4,MedianOfElbows,0.339547,0.712087,0.737382


In [18]:
selectedMethods = ["ElbowOld", "QStemElbow", "MedianOfElbows", "MeanOfElbows", "ThreshholdElbow", "K"]

df = df_backup
df = df[df.power == 2]
df = df[df.treeType == "DCTree"]
df = df[df.core_pts_name == "same_core_pts"]

df = df[df.space_name == "embedding_space"]
# df = df[df.space_name == "original_space"]
df = df[df["partitioningMethod"].isin(selectedMethods)]

df_pivot = pd.pivot_table(
    df,
    values="ari",
    index=["data_name"],
    columns=["partitioningMethod"],
    dropna=False,
    sort=False,
)
df_pivot = df_pivot[selectedMethods]
mean_row = df_pivot.mean(numeric_only=True)
df_pivot = pd.concat([df_pivot, pd.DataFrame([mean_row], index=["Average"])])
df_pivot = (df_pivot * 100).round(2)
df_pivot

partitioningMethod,ElbowOld,QStemElbow,MedianOfElbows,MeanOfElbows,ThreshholdElbow,K
pendigits,95.04,89.99,93.86,93.86,95.03,92.98
optdigits,89.26,85.77,92.97,92.97,92.97,89.46
letterrecognition,11.54,25.06,23.28,21.91,26.98,22.81
easy_blobs,99.03,90.31,99.24,99.24,99.24,99.24
example,87.80,100.00,100.00,100.00,100.00,100.00
USPS,66.28,63.76,63.73,63.73,63.73,67.28
htru,78.58,34.84,78.58,78.57,78.58,90.60
HAR,53.75,52.82,54.23,53.34,52.83,52.99
mice,20.17,19.83,18.86,18.86,20.09,19.38
synth_high,10.51,98.09,82.22,92.01,99.11,99.11


In [19]:
from matplotlib.colors import LinearSegmentedColormap

colors = ["red", "white", "green"]
n_bins = 100  # Number of bins for the colormap

# Create the colormap
cmap = LinearSegmentedColormap.from_list("RedWhiteGreen", colors, N=n_bins)

# Assuming df_pivot is your pivot table
styled = df_pivot.style.background_gradient(
    cmap=cmap, axis=None, vmin=-130, vmax=130
)
# styled = styled.set_table_styles(
#     [{"selector": "td", "props": "border: 1px solid gray;"}, {"selector": "th", "props": "border: 1px solid gray;"}]
# )

# Display in Jupyter Notebook or export to HTML
styled.to_excel("embedding_space.xlsx")
# styled.to_excel("original_space.xlsx")